In [1]:
%%writefile score_rna.py
#!/usr/bin/env python3
"""
score_rna.py — Score a Stanford-RNA-3D-Folding-2 submission against the true
structures fetched from the RCSB PDB, and plot predicted vs. real structures.

WHAT IT REPRODUCES
------------------
The competition metric is the RNA TM-score on C1' atoms: each of your 5 predicted
structures is rigidly superimposed onto the experimental reference (US-align,
sequence-independent), and the target's score is the BEST-of-5 TM-score. The final
score is the mean over targets. TM-score is normalised by the number of resolved
reference residues (L_ref), with the nucleic-acid d0:
        d0 = max(0.3, 0.6 * sqrt(L_ref - 0.5) - 2.5)

TWO SCORING BACKENDS
--------------------
1. --usalign /path/to/USalign   -> the faithful metric. We write C1'-only PDBs and
   call USalign; this is what you should trust for a real leaderboard estimate.
2. (default) a self-contained Python TM-score. It maps predicted<->reference
   residues by SEQUENCE (your prediction has the exact target sequence, so a
   sequence alignment recovers the correct correspondence even when the crystal
   has missing residues), then maximises TM-score over the rigid superposition.
   This closely matches US-align for correctly-predicted sequences; because it
   does not re-search the correspondence the way sequence-independent US-align
   does, it can slightly UNDER-estimate on poor predictions. Good enough to rank
   and debug your pipeline.

USAGE
-----
    python score_rna.py --submission submission.csv --test test_sequences.csv \
                        --out scoring_out
    # optional: --usalign ./USalign   --targets 8ZNQ,9IWF   --no-plots

Requires: numpy, pandas, biopython, requests, matplotlib.
"""

import argparse
import os
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# Chain parsing (same rule the fixed pipeline uses): the scored `sequence` is the
# concatenation of RNA entities, each repeated once per author-chain copy.
# ─────────────────────────────────────────────────────────────────────────────
def parse_entities(all_sequences_str):
    entities = []
    for entry in str(all_sequences_str).strip().split('>'):
        if not entry.strip():
            continue
        parts = entry.split('|')
        n_chains = 0
        for p in parts:
            if 'Chain' in p:
                auth = re.findall(r'\[auth\s+([A-Za-z0-9]+)\]', p)
                if auth:
                    n_chains = len(auth)
                else:
                    m = re.search(r'Chains?\s+([A-Za-z0-9,\s]+)', p)
                    if m:
                        n_chains = len([c for c in re.split(r'[,\s]+', m.group(1)) if c])
                break
        if n_chains == 0:
            n_chains = 1
        seq = re.sub(r'[^A-Za-z]', '', parts[-1]).upper()
        if not seq:
            continue
        is_rna = all(c in 'ACGUN' for c in seq)
        entities.append({'seq': seq, 'n_chains': n_chains, 'is_rna': is_rna})
    return entities

def rna_segments(all_sequences_str, sequence):
    """Ordered list of RNA chain sequences whose concatenation == `sequence`."""
    segs = []
    for e in parse_entities(all_sequences_str):
        if e['is_rna']:
            segs.extend([e['seq']] * e['n_chains'])
    if ''.join(segs) != str(sequence):
        segs = [str(sequence)]        # fall back to single chain if reconstruction differs
    return segs

# ─────────────────────────────────────────────────────────────────────────────
# Reference structure: fetch from RCSB and pull RNA C1' atoms per chain.
# ─────────────────────────────────────────────────────────────────────────────
_DNA = {'DA', 'DC', 'DG', 'DT', 'DU', 'DI'}
_RNA_MOD = {  # a few common modified ribonucleotides -> canonical base
    'PSU': 'U', 'H2U': 'U', '5MU': 'U', '4SU': 'U', '5MC': 'C', 'OMC': 'C',
    '1MA': 'A', '6MZ': 'A', 'MA6': 'A', 'A2M': 'A', '2MG': 'G', '1MG': 'G',
    '7MG': 'G', 'M2G': 'G', 'OMG': 'G', 'YG': 'G', 'G7M': 'G', 'UR3': 'U',
    'OMU': 'U', '5BU': 'U', 'CBV': 'C', 'CCC': 'C', 'GDP': 'G', 'GTP': 'G',
}

def _res_letter(resname):
    r = resname.strip().upper()
    if r in _DNA:
        return None                    # DNA residue -> not RNA
    if r in ('A', 'C', 'G', 'U'):
        return r
    if r in _RNA_MOD:
        return _RNA_MOD[r]
    return 'N'                         # unknown; treat as RNA wildcard

def fetch_reference(pdb_id, cache_dir):
    """Download mmCIF (fallback PDB) for pdb_id; return local path or None."""
    os.makedirs(cache_dir, exist_ok=True)
    for ext, url in ((".cif", f"https://files.rcsb.org/download/{pdb_id}.cif"),
                     (".pdb", f"https://files.rcsb.org/download/{pdb_id}.pdb")):
        path = os.path.join(cache_dir, pdb_id + ext)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200 and len(r.content) > 0:
                with open(path, "wb") as f:
                    f.write(r.content)
                return path
        except Exception as e:
            print(f"    fetch {url} failed: {e}")
    return None

def parse_reference_rna_chains(path):
    """Return {chain_id: {'seq': str, 'coords': (M,3) C1' array}} for RNA chains
       (first model only). Uses biopython; supports .cif and .pdb."""
    from Bio.PDB import MMCIFParser, PDBParser
    parser = MMCIFParser(QUIET=True) if path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("ref", path)
    model = next(iter(structure))
    chains = {}
    for chain in model:
        letters, coords = [], []
        for res in chain:
            if not res.has_id("C1'"):
                continue
            letter = _res_letter(res.get_resname())
            if letter is None:            # DNA residue -> skip this residue
                continue
            letters.append(letter)
            coords.append(res["C1'"].get_coord())
        # RNA chain = has resolved C1' nucleotides and is not DNA-dominated
        if len(letters) >= 3:
            chains[chain.id] = {'seq': ''.join(letters),
                                'coords': np.asarray(coords, dtype=float)}
    return chains

# ─────────────────────────────────────────────────────────────────────────────
# Map predicted (full-sequence) residues to resolved reference residues by
# aligning each expected RNA chain to a reference chain.
# ─────────────────────────────────────────────────────────────────────────────
def _align_map(pred_seq, ref_seq):
    """Global align pred_seq vs ref_seq; return list of (pred_pos, ref_pos) for
       matched columns (both non-gap)."""
    from Bio.Align import PairwiseAligner
    aln = PairwiseAligner()
    aln.mode = 'global'
    aln.match_score, aln.mismatch_score = 2.0, -1.0
    aln.open_gap_score, aln.extend_gap_score = -2.0, -0.5
    a = aln.align(pred_seq, ref_seq)[0]
    pairs = []
    # a.aligned -> two arrays of (start,end) blocks that are aligned 1:1
    for (ps, pe), (rs, re_) in zip(a.aligned[0], a.aligned[1]):
        for k in range(pe - ps):
            pairs.append((ps + k, rs + k))
    return pairs

def build_correspondence(all_sequences, sequence, ref_chains):
    """Return (pred_idx array, ref_coord array) of matched residues across the
       whole target, in global submission order. Greedily assigns each expected
       RNA chain to its best-matching unused reference chain."""
    segs = rna_segments(all_sequences, sequence)
    avail = dict(ref_chains)                 # chain_id -> {seq,coords}
    pred_idx, ref_xyz = [], []
    offset = 0
    for seg in segs:
        # choose the unused reference chain most similar to this segment
        best_id, best_pairs, best_score = None, None, -1
        for cid, info in avail.items():
            pairs = _align_map(seg, info['seq'])
            score = len(pairs) - 0.001 * abs(len(seg) - len(info['seq']))
            if score > best_score:
                best_id, best_pairs, best_score = cid, pairs, score
        if best_id is not None and best_pairs:
            coords = avail[best_id]['coords']
            for (pp, rp) in best_pairs:
                pred_idx.append(offset + pp)
                ref_xyz.append(coords[rp])
            del avail[best_id]
        offset += len(seg)
    if not pred_idx:
        return None, None
    return np.asarray(pred_idx, dtype=int), np.asarray(ref_xyz, dtype=float)

# ─────────────────────────────────────────────────────────────────────────────
# TM-score (Python backend)
# ─────────────────────────────────────────────────────────────────────────────
def _kabsch(P, Q):
    """Rotation+translation mapping P onto Q (least squares)."""
    cP, cQ = P.mean(0), Q.mean(0)
    H = (P - cP).T @ (Q - cQ)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T
    t = cQ - R @ cP
    return R, t

def _d0(L):
    return max(0.3, 0.6 * np.sqrt(max(L - 0.5, 0.0)) - 2.5)

def tm_score(pred, ref):
    """Max TM-score of pred vs ref (1:1 correspondence), normalised by len(ref).
       Returns (tm, pred_superposed_on_ref)."""
    P, Q = np.asarray(pred, float), np.asarray(ref, float)
    L = len(Q)
    d0 = _d0(L)
    R, t = _kabsch(P, Q)
    P2 = (R @ P.T).T + t
    d = np.linalg.norm(P2 - Q, axis=1)
    best_tm, best_P = -1.0, P2
    seeds = [d0 * 0.5, d0, d0 * 1.5, d0 * 2, d0 * 4, 5.0, 10.0]
    for cut in seeds:
        mask = d < cut
        if mask.sum() < 3:
            order = np.argsort(d)[:max(3, L // 5)]
            mask = np.zeros(L, bool); mask[order] = True
        for _ in range(20):
            if mask.sum() < 3:
                break
            R, t = _kabsch(P[mask], Q[mask])
            Pi = (R @ P.T).T + t
            di = np.linalg.norm(Pi - Q, axis=1)
            tm = np.mean(1.0 / (1.0 + (di / d0) ** 2))
            if tm > best_tm:
                best_tm, best_P = tm, Pi
            newmask = di < d0 * 2
            if newmask.sum() < 3:
                order = np.argsort(di)[:max(3, L // 5)]
                newmask = np.zeros(L, bool); newmask[order] = True
            if np.array_equal(newmask, mask):
                break
            mask = newmask
    return float(best_tm), best_P

# ─────────────────────────────────────────────────────────────────────────────
# TM-score (USalign backend, optional)
# ─────────────────────────────────────────────────────────────────────────────
def _write_c1_pdb(coords, seq, path):
    with open(path, "w") as f:
        for i, ((x, y, z), ch) in enumerate(zip(coords, seq), 1):
            f.write(f"ATOM  {i:>5}  C1' {ch:>3} A{i:>4}    "
                    f"{x:>8.3f}{y:>8.3f}{z:>8.3f}  1.00  0.00           C\n")
        for i in range(1, len(coords)):
            f.write(f"CONECT{i:>5}{i+1:>5}\n")
        f.write("END\n")

def tm_score_usalign(usalign, pred_coords, pred_seq, ref_coords, ref_seq, tmp):
    """Return TM-score normalised by the reference (Structure_2) via USalign."""
    import subprocess, re as _re
    p1 = os.path.join(tmp, "pred.pdb"); p2 = os.path.join(tmp, "ref.pdb")
    _write_c1_pdb(pred_coords, pred_seq, p1)
    _write_c1_pdb(ref_coords, ref_seq, p2)

    out = subprocess.run(
        [usalign, p1, p2, "-mol", "RNA", "-atom", " C1'"],
        capture_output=True,
        text=True
    )

    if out.returncode != 0:
        print(f"    USalign failed (code {out.returncode}): {out.stderr.strip()}")
        return None

    # Find the TM-score line normalized by Structure_2 (the reference)
    tm = None
    for line in out.stdout.splitlines():
        if line.startswith("TM-score=") and "Structure_2" in line:
            m = _re.search(r"TM-score=\s*([\d.]+)", line)
            if m:
                tm = float(m.group(1))
            break

    if tm is None:
        print("    could not parse TM-score from USalign output")
    return tm

# ─────────────────────────────────────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────────────────────────────────────
def plot_overlay(target_id, pred_sup, ref, tm, out_png):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa

    fig = plt.figure(figsize=(13, 6))
    # panel 1: overlay (both superposed)
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot(ref[:, 0], ref[:, 1], ref[:, 2], '-', color='#1f77b4', lw=1.8, label='experimental')
    ax1.scatter(ref[:, 0], ref[:, 1], ref[:, 2], s=10, color='#1f77b4')
    ax1.plot(pred_sup[:, 0], pred_sup[:, 1], pred_sup[:, 2], '-', color='#d62728', lw=1.8, label='predicted')
    ax1.scatter(pred_sup[:, 0], pred_sup[:, 1], pred_sup[:, 2], s=10, color='#d62728')
    ax1.set_title(f"{target_id}  —  overlay (TM-score = {tm:.3f})")
    ax1.legend(loc='upper right')
    ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_zlabel("z")
    # panel 2: side by side (offset predicted for clarity)
    ax2 = fig.add_subplot(122, projection="3d")
    span = np.ptp(ref, axis=0).max() if len(ref) else 10.0
    shift = np.array([span * 1.2, 0, 0])
    ax2.plot(ref[:, 0], ref[:, 1], ref[:, 2], '-', color='#1f77b4', lw=1.5)
    p2 = pred_sup + shift
    ax2.plot(p2[:, 0], p2[:, 1], p2[:, 2], '-', color='#d62728', lw=1.5)
    ax2.set_title("experimental (blue)   |   predicted (red)")
    ax2.set_xlabel("x"); ax2.set_ylabel("y"); ax2.set_zlabel("z")
    fig.tight_layout()
    fig.savefig(out_png, dpi=130)
    plt.close(fig)

def plot_summary(df, out_png):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = df.dropna(subset=["tm_score"]).sort_values("tm_score")
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(max(7, 0.35 * len(d)), 5))
    colors = ['#2ca02c' if v >= 0.45 else '#ff7f0e' for v in d["tm_score"]]
    ax.bar(d["target_id"], d["tm_score"], color=colors)
    ax.axhline(0.45, ls='--', color='gray', lw=1, label='0.45 (matching global fold)')
    ax.set_ylabel("best-of-5 TM-score"); ax.set_ylim(0, 1)
    ax.set_title(f"Per-target TM-score  (mean = {d['tm_score'].mean():.3f}, n={len(d)})")
    ax.legend()
    plt.xticks(rotation=90)
    fig.tight_layout()
    fig.savefig(out_png, dpi=130)
    plt.close(fig)

# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
def get_predicted_models(sub_df, target_id, n_res):
    """Return list of 5 (n_res,3) arrays from the submission for this target."""
    rows = sub_df[sub_df["ID"].str.rsplit("_", n=1).str[0] == target_id].copy()
    rows["ridx"] = rows["ID"].str.rsplit("_", n=1).str[1].astype(int)
    rows = rows.sort_values("ridx")
    if len(rows) != n_res:
        print(f"    ⚠️ submission has {len(rows)} rows, expected {n_res}")
    models = []
    for m in range(1, 6):
        models.append(rows[[f"x_{m}", f"y_{m}", f"z_{m}"]].to_numpy(dtype=float))
    return models

def main():
    ap = argparse.ArgumentParser(description="Score an RNA-3D submission vs RCSB truth.")
    ap.add_argument("--submission", required=True)
    ap.add_argument("--test", required=True, help="test_sequences.csv")
    ap.add_argument("--out", default="scoring_out")
    ap.add_argument("--usalign", default=None, help="path to USalign binary (optional)")
    ap.add_argument("--targets", default=None, help="comma-separated subset of target_ids")
    ap.add_argument("--no-plots", action="store_true")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    cache = os.path.join(args.out, "cif_cache")
    plots = os.path.join(args.out, "plots"); os.makedirs(plots, exist_ok=True)

    sub = pd.read_csv(args.submission)
    test = pd.read_csv(args.test)
    if args.targets:
        keep = set(args.targets.split(","))
        test = test[test["target_id"].astype(str).isin(keep)]

    results = []
    for _, row in test.iterrows():
        tid = str(row["target_id"]); seq = str(row["sequence"])
        print(f"\n▶ {tid} (len {len(seq)})")
        rec = {"target_id": tid, "length": len(seq), "tm_score": np.nan, "note": ""}

        ref_path = fetch_reference(tid, cache)
        if not ref_path:
            rec["note"] = "reference fetch failed"; results.append(rec); continue
        try:
            ref_chains = parse_reference_rna_chains(ref_path)
        except Exception as e:
            rec["note"] = f"parse error: {e}"; results.append(rec); continue
        if not ref_chains:
            rec["note"] = "no RNA chains in reference"; results.append(rec); continue

        pred_idx, ref_xyz = build_correspondence(row["all_sequences"], seq, ref_chains)
        if pred_idx is None:
            rec["note"] = "no residue correspondence"; results.append(rec); continue
        rec["n_ref_resolved"] = len(pred_idx)

        models = get_predicted_models(sub, tid, len(seq))
        ref_seq_for_pdb = "".join("A" for _ in ref_xyz)  # resname irrelevant for C1' TM

        best_tm, best_sup, best_ref = -1.0, None, None
        for m, coords in enumerate(models, 1):
            if coords.shape[0] <= max(pred_idx):
                continue
            pred_sub = coords[pred_idx]
            if args.usalign:
                tm = tm_score_usalign(args.usalign, pred_sub, ref_seq_for_pdb,
                                      ref_xyz, ref_seq_for_pdb, args.out)
                # still compute a superposition for plotting
                _, sup = tm_score(pred_sub, ref_xyz)
            else:
                tm, sup = tm_score(pred_sub, ref_xyz)
            if tm is not None and tm > best_tm:
                best_tm, best_sup, best_ref = tm, sup, ref_xyz

        if best_tm < 0:
            rec["note"] = "scoring failed"; results.append(rec); continue
        rec["tm_score"] = best_tm
        print(f"    best-of-5 TM-score = {best_tm:.3f}  (over {len(pred_idx)} resolved residues)")

        

        if not args.no_plots and best_sup is not None:
            try:
                plot_overlay(tid, best_sup, best_ref, best_tm,
                             os.path.join(plots, f"{tid}.png"))
            except Exception as e:
                print(f"    plot failed: {e}")
        
            # NEW: save per-target PDBs for py3Dmol viewing in a notebook cell
            struct_dir = os.path.join(args.out, "structures")
            os.makedirs(struct_dir, exist_ok=True)
            pred_seq_for_pdb = "".join("A" for _ in best_sup)
            ref_seq_for_pdb  = "".join("A" for _ in best_ref)
            _write_c1_pdb(best_sup, pred_seq_for_pdb, os.path.join(struct_dir, f"{tid}_pred.pdb"))
            _write_c1_pdb(best_ref, ref_seq_for_pdb,  os.path.join(struct_dir, f"{tid}_ref.pdb"))
            
        results.append(rec)

    df = pd.DataFrame(results)
    csv_path = os.path.join(args.out, "tm_scores.csv")
    df.to_csv(csv_path, index=False)
    if not args.no_plots:
        plot_summary(df, os.path.join(args.out, "tm_scores_summary.png"))

    scored = df.dropna(subset=["tm_score"])
    print("\n" + "=" * 55)
    print(f"Scored {len(scored)}/{len(df)} targets")
    if len(scored):
        print(f"MEAN best-of-5 TM-score = {scored['tm_score'].mean():.4f}")
    print(f"Per-target table -> {csv_path}")
    if not args.no_plots:
        print(f"Plots -> {os.path.join(args.out, 'plots')}/  and  tm_scores_summary.png")

if __name__ == "__main__":
    main()


Writing score_rna.py


In [2]:
!pip install biopython
!pip install py3Dmol

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.0 MB/s eta 0:00:00


In [3]:
!g++ -O3 -ffast-math -o USalign /kaggle/input/datasets/khushvaghela/test-submission/USalign-master/USalign-master/USalign.cpp

In [4]:
!ls -l
!file USalign

total 1060
---------- 1 root root   36664 Jul  8 07:41 __notebook__.ipynb
-rw-r--r-- 1 root root   21759 Jul  8 07:40 score_rna.py
-rwxr-xr-x 1 root root 1021136 Jul  8 07:41 USalign
USalign: ELF 64-bit LSB pie executable, x86-64, version 1 (SYSV), dynamically linked, interpreter /lib64/ld-linux-x86-64.so.2, BuildID[sha1]=374d055a249e4050984c8a0c2577eda852bd607d, for GNU/Linux 3.2.0, with debug_info, not stripped


In [5]:
!python score_rna.py \
    --submission "/kaggle/input/datasets/khushvaghela/test-submission/final_submission.csv" \
    --test "/kaggle/input/datasets/khushvaghela/test-submission/final_test_sequences.csv" \
    --out scoring_out \
    --usalign /kaggle/working/USalign


▶ 8ZNQ (len 30)
    best-of-5 TM-score = 0.211  (over 30 resolved residues)

▶ 9IWF (len 69)
    best-of-5 TM-score = 0.607  (over 69 resolved residues)

▶ 9JGM (len 210)
    best-of-5 TM-score = 0.165  (over 210 resolved residues)

▶ 9MME (len 4640)
    best-of-5 TM-score = 0.125  (over 4168 resolved residues)

▶ 9J09 (len 214)
    best-of-5 TM-score = 0.105  (over 114 resolved residues)

▶ 9E9Q (len 101)
    best-of-5 TM-score = 0.490  (over 101 resolved residues)

▶ 9CFN (len 59)
    best-of-5 TM-score = 0.213  (over 55 resolved residues)

▶ 9OBM (len 73)
    best-of-5 TM-score = 0.573  (over 73 resolved residues)

▶ 9G4P (len 68)
    best-of-5 TM-score = 0.165  (over 66 resolved residues)

▶ 9G4Q (len 104)
    best-of-5 TM-score = 0.157  (over 104 resolved residues)

▶ 9G4R (len 47)
    best-of-5 TM-score = 0.289  (over 47 resolved residues)

▶ 9RVP (len 34)
    best-of-5 TM-score = 0.360  (over 30 resolved residues)

▶ 9JFS (len 246)
    best-of-5 TM-score = 0.143  (over 138 reso

In [6]:
import pandas as pd
import py3Dmol
from IPython.display import display, HTML

def show_all_overlays(out_dir="scoring_out"):
    scores_df = pd.read_csv(f"{out_dir}/tm_scores.csv")

    for _, row in scores_df.iterrows():
        tid = row["target_id"]
        tm = row["tm_score"]

        pred_path = f"{out_dir}/structures/{tid}_pred.pdb"
        ref_path  = f"{out_dir}/structures/{tid}_ref.pdb"

        # skip targets that failed to score / have no saved structures
        try:
            with open(pred_path) as f:
                pred_pdb = f.read()
            with open(ref_path) as f:
                ref_pdb = f.read()
        except FileNotFoundError:
            print(f"⚠️  {tid}: no structure files found (note: {row.get('note', '')})")
            continue

        score_str = f"{tm:.3f}" if pd.notna(tm) else "N/A"
        print(f"▶ {tid}  —  best-of-5 TM-score = {score_str}")

        view = py3Dmol.view(width=900, height=550)

        view.addModel(ref_pdb, "pdb")
        view.setStyle({"model": 0}, {
            "stick":  {"color": "#1f77b4", "radius": 0.35, "opacity": 0.9},
            "sphere": {"color": "#1f77b4", "scale": 0.3}
        })

        view.addModel(pred_pdb, "pdb")
        view.setStyle({"model": 1}, {
            "stick":  {"color": "#d62728", "radius": 0.35, "opacity": 0.9},
            "sphere": {"color": "#d62728", "scale": 0.3}
        })

        view.setBackgroundColor("0xf7f7f7")
        view.zoomTo()

        display(HTML(f"""
        <div style="font-family: sans-serif; margin-bottom: 6px;">
          <b>{tid}</b> &nbsp;(TM-score = {score_str})<br>
          <span style="color:#1f77b4;">&#9679;</span> experimental &nbsp;&nbsp;
          <span style="color:#d62728;">&#9679;</span> predicted
        </div>
        """))
        view.show()

show_all_overlays()

▶ 8ZNQ  —  best-of-5 TM-score = 0.211


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

▶ 9IWF  —  best-of-5 TM-score = 0.607


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

▶ 9JGM  —  best-of-5 TM-score = 0.165


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

▶ 9LJN  —  best-of-5 TM-score = 0.635


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

▶ 9ZCC  —  best-of-5 TM-score = 0.161


3Dmol.js failed to load for some reason. Please check your browser console for error messages.